# SQL practice 6: window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-06-window-functions.ipynb)

14 exercises on ranking, top-N, running and moving aggregates, LAG/LEAD, forward fill and streaks. Each exercise names the [reference entries](sql-reference-06-window-functions.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 6](sql-reference-06-window-functions.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

7 easy, 6 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Salary rank within department](#exercise-1-salary-rank-within-department) | Easy | [`ROW_NUMBER`, `RANK`, `DENSE_RANK`](sql-reference-06-window-functions.ipynb#row_number-rank-dense_rank) |
| 2 | [Top two orders per customer](#exercise-2-top-two-orders-per-customer) | Easy | [Top-N per group and deduplication](sql-reference-06-window-functions.ipynb#top-n-per-group-and-deduplication), [Filtering on a window result](sql-reference-06-window-functions.ipynb#filtering-on-a-window-result) |
| 3 | [Second-highest distinct salary](#exercise-3-second-highest-distinct-salary) | Easy | [`ROW_NUMBER`, `RANK`, `DENSE_RANK`](sql-reference-06-window-functions.ipynb#row_number-rank-dense_rank) |
| 4 | [Running revenue](#exercise-4-running-revenue) | Easy | [Running totals and the default frame](sql-reference-06-window-functions.ipynb#running-totals-and-the-default-frame) |
| 5 | [Share of each customer's spend](#exercise-5-share-of-each-customers-spend) | Easy | [The `OVER` clause](sql-reference-06-window-functions.ipynb#the-over-clause) |
| 6 | [Day-over-day revenue change](#exercise-6-day-over-day-revenue-change) | Medium | [`LAG` and `LEAD`](sql-reference-06-window-functions.ipynb#lag-and-lead) |
| 7 | [Moving average of the last three orders](#exercise-7-moving-average-of-the-last-three-orders) | Medium | [Frames: `ROWS` and `RANGE`](sql-reference-06-window-functions.ipynb#frames-rows-and-range) |
| 8 | [Remove exact duplicate events](#exercise-8-remove-exact-duplicate-events) | Easy | [Top-N per group and deduplication](sql-reference-06-window-functions.ipynb#top-n-per-group-and-deduplication) |
| 9 | [First and last status per customer](#exercise-9-first-and-last-status-per-customer) | Medium | [`FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`](sql-reference-06-window-functions.ipynb#first_value-last_value-nth_value) |
| 10 | [Quartiles of order value](#exercise-10-quartiles-of-order-value) | Easy | [`NTILE`, `PERCENT_RANK`, `CUME_DIST`](sql-reference-06-window-functions.ipynb#ntile-percent_rank-cume_dist) |
| 11 | [Minutes since the previous event](#exercise-11-minutes-since-the-previous-event) | Medium | [`LAG` and `LEAD`](sql-reference-06-window-functions.ipynb#lag-and-lead) |
| 12 | [Forward-fill sensor readings](#exercise-12-forward-fill-sensor-readings) | Medium | [`FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`](sql-reference-06-window-functions.ipynb#first_value-last_value-nth_value) |
| 13 | [Running count of distinct products](#exercise-13-running-count-of-distinct-products) | Medium | [Running totals and the default frame](sql-reference-06-window-functions.ipynb#running-totals-and-the-default-frame) |
| 14 | [Longest login streak](#exercise-14-longest-login-streak) | Hard | [Gaps and islands](sql-reference-06-window-functions.ipynb#gaps-and-islands) |

## Exercise 1: Salary rank within department

**Difficulty:** Easy · about 5 min · **Practises:** [`ROW_NUMBER`, `RANK`, `DENSE_RANK`](sql-reference-06-window-functions.ipynb#row_number-rank-dense_rank)

*Ranking within groups is the most common window question.*

Return `name`, `dept`, `salary` and `dept_rank` (dense rank by salary, highest first) for employees with a department, ordered by `dept`, `dept_rank`, `name`.

**Expected output**

| name | dept | salary | dept_rank |
| --- | --- | --- | --- |
| Ada | Engineering | 185000 | 1 |
| Dana | Engineering | 155000 | 2 |
| Brett | Engineering | 150000 | 3 |
| Chen | Engineering | 150000 | 3 |
| Hana | Finance | 140000 | 1 |
| Ivan | Finance | 88000 | 2 |
| Eli | Sales | 130000 | 1 |
| Farah | Sales | 95000 | 2 |
| Gus | Sales | 95000 | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT name, dept, salary,
           dense_rank() OVER (PARTITION BY dept ORDER BY salary DESC) AS dept_rank
    FROM employees
    WHERE dept IS NOT NULL
    ORDER BY dept, dept_rank, name
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Top two orders per customer

**Difficulty:** Easy · about 5 min · **Practises:** [Top-N per group and deduplication](sql-reference-06-window-functions.ipynb#top-n-per-group-and-deduplication), [Filtering on a window result](sql-reference-06-window-functions.ipynb#filtering-on-a-window-result)

*Top-N per group without QUALIFY.*

Return `customer_id`, `order_id` and `amount` for each customer's two largest orders (ties by smaller `order_id`), ordered by `customer_id`, then `amount` descending.

**Expected output**

| customer_id | order_id | amount |
| --- | --- | --- |
| 1 | 1 | 120.00 |
| 1 | 2 | 84.50 |
| 2 | 4 | 260.00 |
| 2 | 5 | 55.25 |
| 3 | 7 | 310.00 |
| 3 | 6 | 149.00 |
| 4 | 9 | 1180.00 |
| 4 | 8 | 89.90 |
| 5 | 10 | 74.00 |
| 6 | 11 | 210.00 |
| 6 | 12 | 96.40 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, order_id, amount
    FROM (
        SELECT customer_id, order_id, amount,
               row_number() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
        FROM orders
    )
    WHERE rn <= 2
    ORDER BY customer_id, amount DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Second-highest distinct salary

**Difficulty:** Easy · about 5 min · **Practises:** [`ROW_NUMBER`, `RANK`, `DENSE_RANK`](sql-reference-06-window-functions.ipynb#row_number-rank-dense_rank)

*dense_rank counts values, not rows, so ties do not shift the answer.*

Return the `name` and `salary` of every employee earning the second-highest distinct salary, ordered by `name`.

**Expected output**

| name | salary |
| --- | --- |
| Dana | 155000 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT name, salary
    FROM (SELECT name, salary, dense_rank() OVER (ORDER BY salary DESC) AS rnk FROM employees)
    WHERE rnk = 2
    ORDER BY name
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Running revenue

**Difficulty:** Easy · about 5 min · **Practises:** [Running totals and the default frame](sql-reference-06-window-functions.ipynb#running-totals-and-the-default-frame)

*Cumulative metrics for burn-up charts.*

Return `order_id`, `order_ts`, `amount` and `running_revenue` (cumulative sum in `order_ts` order, row by row), ordered by `order_ts`.

**Expected output**

| order_id | order_ts | amount | running_revenue |
| --- | --- | --- | --- |
| 1 | 2026-01-05 09:15:00 | 120.00 | 120.00 |
| 2 | 2026-01-05 10:05:00 | 84.50 | 204.50 |
| 3 | 2026-01-05 11:40:00 | 30.00 | 234.50 |
| 4 | 2026-01-06 15:20:00 | 260.00 | 494.50 |
| 5 | 2026-01-07 08:05:00 | 55.25 | 549.75 |
| 6 | 2026-01-07 17:45:00 | 149.00 | 698.75 |
| 7 | 2026-01-09 12:10:00 | 310.00 | 1008.75 |
| 8 | 2026-01-10 19:30:00 | 89.90 | 1098.65 |
| 9 | 2026-01-12 08:55:00 | 1180.00 | 2278.65 |
| 10 | 2026-01-12 13:00:00 | 74.00 | 2352.65 |
| 11 | 2026-01-13 10:25:00 | 210.00 | 2562.65 |
| 12 | 2026-01-14 00:30:00 | 96.40 | 2659.05 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, order_ts, amount,
           sum(amount) OVER (ORDER BY order_ts, order_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue
    FROM orders
    ORDER BY order_ts
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Share of each customer's spend

**Difficulty:** Easy · about 5 min · **Practises:** [The `OVER` clause](sql-reference-06-window-functions.ipynb#the-over-clause)

*Percent of total without a self join.*

Return `customer_id`, `order_id`, `amount` and `pct` (the order's share of that customer's total, as a percentage rounded to 1) for customers 1 to 3, ordered by `customer_id`, `order_id`.

**Expected output**

| customer_id | order_id | amount | pct |
| --- | --- | --- | --- |
| 1 | 1 | 120.00 | 51.2 |
| 1 | 2 | 84.50 | 36.0 |
| 1 | 3 | 30.00 | 12.8 |
| 2 | 4 | 260.00 | 82.5 |
| 2 | 5 | 55.25 | 17.5 |
| 3 | 6 | 149.00 | 32.5 |
| 3 | 7 | 310.00 | 67.5 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, order_id, amount,
           round(100 * amount / sum(amount) OVER (PARTITION BY customer_id), 1) AS pct
    FROM orders
    WHERE customer_id <= 3
    ORDER BY customer_id, order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Day-over-day revenue change

**Difficulty:** Medium · about 10 min · **Practises:** [`LAG` and `LEAD`](sql-reference-06-window-functions.ipynb#lag-and-lead)

*Period-over-period deltas drive alerting on revenue drops.*

Per calendar day of `order_ts`, return `day`, `revenue`, `prev_revenue` (previous day **with orders**) and `pct_change` (rounded to 1, NULL on the first day). Order by `day`.

**Expected output**

| day | revenue | prev_revenue | pct_change |
| --- | --- | --- | --- |
| 2026-01-05 | 234.50 | null | null |
| 2026-01-06 | 260.00 | 234.50 | 10.9 |
| 2026-01-07 | 204.25 | 260.00 | -21.4 |
| 2026-01-09 | 310.00 | 204.25 | 51.8 |
| 2026-01-10 | 89.90 | 310.00 | -71.0 |
| 2026-01-12 | 1254.00 | 89.90 | 1294.9 |
| 2026-01-13 | 210.00 | 1254.00 | -83.3 |
| 2026-01-14 | 96.40 | 210.00 | -54.1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH daily AS (
        SELECT to_date(order_ts) AS day, sum(amount) AS revenue
        FROM orders
        GROUP BY to_date(order_ts)
    )
    SELECT day, revenue,
           lag(revenue) OVER (ORDER BY day) AS prev_revenue,
           round(100 * (revenue - lag(revenue) OVER (ORDER BY day)) / lag(revenue) OVER (ORDER BY day), 1) AS pct_change
    FROM daily
    ORDER BY day
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Moving average of the last three orders

**Difficulty:** Medium · about 5 min · **Practises:** [Frames: `ROWS` and `RANGE`](sql-reference-06-window-functions.ipynb#frames-rows-and-range)

*Smoothing noisy series per entity.*

Return `customer_id`, `order_id` and `moving_avg`: the average amount of the customer's current and up to two previous orders (by `order_ts`), rounded to 2. Customers 1 to 4, ordered by `customer_id`, `order_ts`.

**Expected output**

| customer_id | order_id | moving_avg |
| --- | --- | --- |
| 1 | 1 | 120.00 |
| 1 | 2 | 102.25 |
| 1 | 3 | 78.17 |
| 2 | 4 | 260.00 |
| 2 | 5 | 157.63 |
| 3 | 6 | 149.00 |
| 3 | 7 | 229.50 |
| 4 | 8 | 89.90 |
| 4 | 9 | 634.95 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, order_id,
           round(avg(amount) OVER (PARTITION BY customer_id ORDER BY order_ts
                                   ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg
    FROM orders
    WHERE customer_id <= 4
    ORDER BY customer_id, order_ts
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Remove exact duplicate events

**Difficulty:** Easy · about 5 min · **Practises:** [Top-N per group and deduplication](sql-reference-06-window-functions.ipynb#top-n-per-group-and-deduplication)

*At-least-once delivery writes the same event twice.*

Return `user_id`, `event` and `ts` for events with exact duplicates removed, using `row_number`, ordered by `user_id`, `ts`, `event`.

**Expected output**

| user_id | event | ts |
| --- | --- | --- |
| 1 | view | 2026-01-06 09:00:00 |
| 1 | add_to_cart | 2026-01-06 09:02:00 |
| 1 | purchase | 2026-01-06 09:10:00 |
| 2 | view | 2026-01-07 10:00:00 |
| 2 | purchase | 2026-01-07 10:40:00 |
| 3 | view | 2026-01-06 11:15:00 |
| 3 | view | 2026-01-08 11:20:00 |
| 4 | add_to_cart | 2026-01-09 20:05:00 |
| 4 | purchase | 2026-01-10 08:30:00 |
| 5 | view | 2026-01-11 14:45:00 |
| 5 | view | 2026-01-12 09:00:00 |
| 5 | purchase | 2026-01-12 12:30:00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT user_id, event, ts
    FROM (
        SELECT user_id, event, ts,
               row_number() OVER (PARTITION BY user_id, event, ts ORDER BY ts) AS copy
        FROM events
    )
    WHERE copy = 1
    ORDER BY user_id, ts, event
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: First and last status per customer

**Difficulty:** Medium · about 10 min · **Practises:** [`FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`](sql-reference-06-window-functions.ipynb#first_value-last_value-nth_value)

*last_value with the default frame is a classic bug.*

Return one row per `customer_id` with `first_status` and `last_status` (by `order_ts`) and `orders`, ordered by `customer_id`. Use window functions, then remove duplicates.

**Expected output**

| customer_id | first_status | last_status | orders |
| --- | --- | --- | --- |
| 1 | paid | returned | 3 |
| 2 | paid | placed | 2 |
| 3 | shipped | paid | 2 |
| 4 | returned | shipped | 2 |
| 5 | paid | paid | 1 |
| 6 | placed | shipped | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT DISTINCT customer_id,
           first_value(status) OVER w AS first_status,
           last_value(status)  OVER w AS last_status,
           count(*)            OVER w AS orders
    FROM orders
    WINDOW w AS (PARTITION BY customer_id ORDER BY order_ts
                 ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Quartiles of order value

**Difficulty:** Easy · about 3 min · **Practises:** [`NTILE`, `PERCENT_RANK`, `CUME_DIST`](sql-reference-06-window-functions.ipynb#ntile-percent_rank-cume_dist)

*Bucketing customers or orders into equal-sized groups.*

Return `order_id`, `amount` and `quartile` (1 = smallest, by amount then `order_id`), ordered by `order_id`.

**Expected output**

| order_id | amount | quartile |
| --- | --- | --- |
| 1 | 120.00 | 3 |
| 2 | 84.50 | 2 |
| 3 | 30.00 | 1 |
| 4 | 260.00 | 4 |
| 5 | 55.25 | 1 |
| 6 | 149.00 | 3 |
| 7 | 310.00 | 4 |
| 8 | 89.90 | 2 |
| 9 | 1180.00 | 4 |
| 10 | 74.00 | 1 |
| 11 | 210.00 | 3 |
| 12 | 96.40 | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, amount, ntile(4) OVER (ORDER BY amount, order_id) AS quartile
    FROM orders
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Minutes since the previous event

**Difficulty:** Medium · about 5 min · **Practises:** [`LAG` and `LEAD`](sql-reference-06-window-functions.ipynb#lag-and-lead)

*Inter-event gaps feed sessionisation and anomaly detection.*

Return `user_id`, `event`, `ts` and `minutes_since_prev` (whole minutes since the user's previous event, NULL for the first) for users 1 to 3, ordered by `user_id`, `ts`.

**Expected output**

| user_id | event | ts | minutes_since_prev |
| --- | --- | --- | --- |
| 1 | view | 2026-01-06 09:00:00 | null |
| 1 | add_to_cart | 2026-01-06 09:02:00 | 2 |
| 1 | purchase | 2026-01-06 09:10:00 | 8 |
| 2 | view | 2026-01-07 10:00:00 | null |
| 2 | purchase | 2026-01-07 10:40:00 | 40 |
| 3 | view | 2026-01-06 11:15:00 | null |
| 3 | view | 2026-01-08 11:20:00 | 2885 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT user_id, event, ts,
           CAST((unix_seconds(ts) - unix_seconds(lag(ts) OVER (PARTITION BY user_id ORDER BY ts))) / 60 AS BIGINT) AS minutes_since_prev
    FROM events
    WHERE user_id <= 3
    ORDER BY user_id, ts
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Forward-fill sensor readings

**Difficulty:** Medium · about 10 min · **Practises:** [`FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`](sql-reference-06-window-functions.ipynb#first_value-last_value-nth_value)

*Sparse telemetry is filled with the last known value per device.*

Using `readings` below, return `device`, `minute`, `temp` and `filled` (the latest non-NULL temperature so far for that device; NULL until the first reading), ordered by `device`, `minute`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW readings AS SELECT * FROM VALUES
  ('a', 1, 20.5), ('a', 2, NULL), ('a', 3, NULL), ('a', 4, 21.0), ('a', 5, NULL),
  ('b', 1, NULL), ('b', 2, 18.0), ('b', 3, NULL)
AS readings(device, minute, temp)
""")

**Expected output**

| device | minute | temp | filled |
| --- | --- | --- | --- |
| a | 1 | 20.5 | 20.5 |
| a | 2 | null | 20.5 |
| a | 3 | null | 20.5 |
| a | 4 | 21.0 | 21.0 |
| a | 5 | null | 21.0 |
| b | 1 | null | null |
| b | 2 | 18.0 | 18.0 |
| b | 3 | null | 18.0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT device, minute, temp,
           last_value(temp) IGNORE NULLS OVER (PARTITION BY device ORDER BY minute) AS filled
    FROM readings
    ORDER BY device, minute
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Running count of distinct products

**Difficulty:** Medium · about 10 min · **Practises:** [Running totals and the default frame](sql-reference-06-window-functions.ipynb#running-totals-and-the-default-frame)

*count(DISTINCT) is not allowed in Spark windows; collect_set is the workaround.*

Return `customer_id`, `order_id` and `distinct_skus_so_far` (distinct SKUs the customer has bought up to and including this order, by `order_ts`) for customers 1 to 3, ordered by `customer_id`, `order_id`.

**Expected output**

| customer_id | order_id | distinct_skus_so_far |
| --- | --- | --- |
| 1 | 1 | 2 |
| 1 | 2 | 4 |
| 1 | 3 | 4 |
| 2 | 4 | 2 |
| 2 | 5 | 3 |
| 3 | 6 | 1 |
| 3 | 7 | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH lines AS (
        SELECT o.customer_id, o.order_id, o.order_ts, i.sku
        FROM orders o JOIN order_items i ON i.order_id = o.order_id
        WHERE o.customer_id <= 3
    ),
    running AS (
        SELECT customer_id, order_id,
               size(collect_set(sku) OVER (PARTITION BY customer_id ORDER BY order_ts)) AS distinct_skus_so_far
        FROM lines
    )
    SELECT DISTINCT customer_id, order_id, distinct_skus_so_far
    FROM running
    ORDER BY customer_id, order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Longest login streak

**Difficulty:** Hard · about 20 min · **Practises:** [Gaps and islands](sql-reference-06-window-functions.ipynb#gaps-and-islands)

*Consecutive-day streaks are the canonical gaps-and-islands interview question.*

Using `logins` below (several logins per day possible), return `user_id`, `streak_start`, `streak_end` and `days` for each user's **longest** run of consecutive login days (earliest run on ties), ordered by `user_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW logins AS SELECT * FROM VALUES
  (1, TIMESTAMP'2026-01-01 08:00:00'), (1, TIMESTAMP'2026-01-02 09:00:00'), (1, TIMESTAMP'2026-01-02 21:00:00'),
  (1, TIMESTAMP'2026-01-03 07:30:00'), (1, TIMESTAMP'2026-01-05 10:00:00'), (1, TIMESTAMP'2026-01-06 10:00:00'),
  (2, TIMESTAMP'2026-01-10 12:00:00'), (2, TIMESTAMP'2026-01-12 12:00:00'),
  (3, TIMESTAMP'2026-01-30 23:00:00'), (3, TIMESTAMP'2026-01-31 01:00:00'), (3, TIMESTAMP'2026-02-01 01:00:00')
AS logins(user_id, login_ts)
""")

**Expected output**

| user_id | streak_start | streak_end | days |
| --- | --- | --- | --- |
| 1 | 2026-01-01 | 2026-01-03 | 3 |
| 2 | 2026-01-10 | 2026-01-10 | 1 |
| 3 | 2026-01-30 | 2026-02-01 | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH days AS (
        SELECT DISTINCT user_id, to_date(login_ts) AS day FROM logins
    ),
    islands AS (
        SELECT user_id, day,
               date_sub(day, CAST(row_number() OVER (PARTITION BY user_id ORDER BY day) AS INT)) AS island
        FROM days
    ),
    streaks AS (
        SELECT user_id, min(day) AS streak_start, max(day) AS streak_end, count(*) AS days
        FROM islands
        GROUP BY user_id, island
    )
    SELECT user_id, streak_start, streak_end, days
    FROM (SELECT *, row_number() OVER (PARTITION BY user_id ORDER BY days DESC, streak_start) AS rn FROM streaks)
    WHERE rn = 1
    ORDER BY user_id
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 6: window functions](sql-reference-06-window-functions.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).